## User-Based + Item-Based

![ModelPredict](ModelPredict.png)

In [34]:
import ast
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

### Load Steam Dataset

In [3]:
# Подгрузим Steam Dataset, возьмем только записи тех кто уже поиграл в игру.

steam_raw = pd.read_csv('../SteamDataset200K/steam-200k.csv', header=None, names=['user_id', 'game', 'behavior', 'hours', 'zero'])

df = steam_raw[steam_raw['behavior'] == 'play'].copy()
df = df[['user_id', 'game', 'hours']].reset_index(drop=True)

df.head()

,user_id,game,hours
0,151603712,The Elder Scrolls V Skyrim,273.0
1,151603712,Fallout 4,87.0
2,151603712,Spore,14.9
3,151603712,Fallout New Vegas,12.1
4,151603712,Left 4 Dead 2,8.9


In [7]:
# Конвертируем игровые часы в рейтинг, чтобы метрика стала более читаемой.
# Масштабируем логарифмически.

def hours_to_rating(hours_series):
    log_h = np.log1p(hours_series)
    scaler = MinMaxScaler(feature_range=(1, 10))
    ratings = scaler.fit_transform(log_h.values.reshape(-1, 1)).flatten()

    return np.round(ratings, 2)


df['rating'] = hours_to_rating(df['hours'])

df.head(10)

,user_id,game,hours,rating
0,151603712,The Elder Scrolls V Skyrim,273.0,6.35
1,151603712,Fallout 4,87.0,5.25
2,151603712,Spore,14.9,3.59
3,151603712,Fallout New Vegas,12.1,3.40
4,151603712,Left 4 Dead 2,8.9,3.13
5,151603712,HuniePop,8.5,3.09
6,151603712,Path of Exile,8.1,3.05
7,151603712,Poly Bridge,7.5,2.98
8,151603712,Left 4 Dead,3.3,2.32
9,151603712,Team Fortress 2,2.8,2.20


In [8]:
# Делаем датасет менее разряженным (это может повлечь проблемы: непопулярные игры не будут рекомендоваться, но мы подумаем над этим позже)

MIN_GAMES_PER_USER = 5
MIN_USERS_PER_GAME = 10

user_counts = df.groupby('user_id')['game'].count()
game_counts = df.groupby('game')['user_id'].count()

active_users = user_counts[user_counts >= MIN_GAMES_PER_USER].index
popular_games = game_counts[game_counts >= MIN_USERS_PER_GAME].index

df_filtered = df[df['user_id'].isin(active_users) & df['game'].isin(popular_games)]

user_item_matrix = df_filtered.pivot_table(
    index='user_id',
    columns='game',
    values='rating',
    aggfunc='mean'
).fillna(0)

user_item_matrix

game,3DMark,7 Days to Die,8BitMMO,A Game of Thrones - Genesis,A Virus Named TOM,A.V.A - Alliance of Valiant Arms,ACE - Arena Cyber Evolution,ACE COMBAT ASSAULT HORIZON Enhanced Edition,APB Reloaded,ARK Survival Evolved,...,Xenonauts,Yet Another Zombie Defense,You Have to Win the Game,You Need A Budget 4 (YNAB),Zeno Clash,Zombie Panic Source,Zombies Monsters Robots,resident evil 4 / biohazard 4,sZone-Online,theHunter
user_id,,,,,,,,,,,,,,,,,,,,,
5250,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76767,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86540,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229911,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
298950,0.0,3.55,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.53,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303129589,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
303525289,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
304971849,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### User-Based

In [42]:
def user_based_rec(user_id, user_item_matrix, n_neighbors=20, top_n=20):
    if user_id not in user_item_matrix.index:
        return None

    sim_matrix = cosine_similarity(user_item_matrix.values)
    sim_df = pd.DataFrame(sim_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)

    user_sim = sim_df[user_id].drop(user_id).sort_values(ascending=False)
    neighbors = user_sim.head(n_neighbors)

    user_games = user_item_matrix.loc[user_id]
    unplayed = user_games[user_games == 0].index

    neighbor_ratings = user_item_matrix.loc[neighbors.index, unplayed]
    weights = neighbors.values.reshape(-1, 1)

    weighted_sum = (neighbor_ratings * weights).sum(axis=0)
    weight_total = (neighbor_ratings > 0).astype(int).T.dot(weights.flatten())
    weight_total = weight_total.replace(0, np.nan)

    predicted = (weighted_sum / weight_total).dropna().sort_values(ascending=False)

    result = predicted.head(top_n).reset_index()
    result.columns = ['game', 'ub_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]
ub_recs = user_based_rec(sample_user, user_item_matrix)

played, ub_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                            game  ub_score
 0                              Source Filmmaker  5.830000
 1                          Farming Simulator 15  5.250000
 2                          Realm of the Mad God  5.110000
 3                                Clicker Heroes  5.020000
 4                            Tabletop Simulator  4.860000
 5                                    Watch_Dogs  4.780000
 6                                 Two Worlds II  4.620000
 7   Grand Theft Auto Episodes from Liberty City  4.530000
 8       Call of Duty Black Ops II - Multiplayer  4.443182
 9                                      Terraria  4.375847
 10                South Park The Stick of Truth  4.300000
 11                       Euro Truck Simulator 2  4.2

### Load PlayMyData

In [29]:
pc = pd.read_csv('../PlayMyData/all_games_PC.csv')
ps = pd.read_csv('../PlayMyData/all_games_PlayStation.csv')
genres_df = pd.read_csv('../PlayMyData/genres.csv')

games_df = pd.concat([pc, ps], ignore_index=True).drop_duplicates(subset='id').reset_index(drop=True)

games_df['rating'] = pd.to_numeric(games_df['rating'], errors='coerce')

games_df.head()

,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,review_score,review_count,people_polled
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
def parse_genres(value):
    if not isinstance(value, str):
        return []
    value = value.strip()
    if value == '' or value == 'Missing':
        return []
    try:
        result = ast.literal_eval(value)

        return result if isinstance(result, list) else []
    except Exception:
        return []


games_df['genres_list'] = games_df['genres'].apply(parse_genres)

#One-hot encoding
all_genre_ids = genres_df['genre_id'].tolist()
for gid in all_genre_ids:
    games_df[f'genre_{gid}'] = games_df['genres_list'].apply(lambda x: int(gid in x))

num_features = ['rating', 'review_score', 'main', 'extra', 'completionist']
for column in num_features:
    games_df[column] = pd.to_numeric(games_df[column], errors='coerce')
    games_df[column] = games_df[column].fillna(games_df[column].median())
    max_value = games_df[column].max()

    if max_value > 0:
        games_df[column] = games_df[column] / max_value

games_df.head(10)

['0x5b', '0x35', '0x5d']


,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,...,genre_26,genre_25,genre_30,genre_31,genre_33,genre_34,genre_32,genre_35,genre_36,genre_2
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,0.7,0.000309,0.0,0.0,...,1,0,0,1,0,0,0,0,0,0
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
5,[10],204650,Rage Rally,[6],Missing,Missing,0.7,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,[34],194713,Ripple: Blue Seal he Youkoso,[6],Main character loses a job and applies for an ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,1,0,0,0,0
7,[31],73345,The Mystery of the Nautilus,[6],This adventure game was inspired by the Jules ...,The story begins aboard the USS Shark a resear...,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
8,"[2, 31]",71917,JumpStart: Animal Adventures,"[6, 14]",Educational game for teaching kids about wild ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,1
9,Missing,61935,Farland Symphony,[6],A spinoff title in the Farland Story RPG franc...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [36]:
steam_games_names = set(user_item_matrix.columns)
content_games = games_df[games_df['name'].isin(steam_games_names)].copy()
content_games = content_games.set_index('name')

genre_columns = [f'genre_{gid}' for gid in all_genre_ids]
num_columns = ['rating', 'review_score', 'main', 'extra', 'completionist']
feature_columns = genre_columns + num_columns

GENRE_WEIGHT = 3.0
NUMERIC_WEIGHT = 1.5

feature_matrix = content_games[feature_columns].fillna(0).values.copy().astype(float)
n_genre = len(genre_columns)

feature_matrix[:, :n_genre] *= GENRE_WEIGHT
feature_matrix[:, :n_genre] *= NUMERIC_WEIGHT

item_sim_matrix = cosine_similarity(feature_matrix)
item_sim_df = pd.DataFrame(
    item_sim_matrix,
    index = content_games.index,
    columns=content_games.index
)

item_sim_df

name,Dungeon Siege,Starbound,Poly Bridge,SolForge,Dishonored,"Cook, Serve, Delicious!",Hotline Miami,Euro Truck Simulator 2,Primal Carnage,Natural Selection 2,...,Sid Meier's Pirates!,DayZ,Unturned,Five Nights at Freddy's 4,Mass Effect 2,Mass Effect,L.A. Noire,The Forest,Fuse,Five Nights at Freddy's 2
name,,,,,,,,,,,,,,,,,,,,,
Dungeon Siege,1.000000,0.509293,0.029182,0.504135,0.590109,0.029666,0.034549,0.033152,0.034728,0.022960,...,0.456599,0.411252,0.026508,0.028471,0.514028,0.510655,0.035667,0.025410,0.033429,0.029065
Starbound,0.509293,1.000000,0.298665,0.257014,0.584054,0.260707,0.300346,0.299931,0.363990,0.232147,...,0.453606,0.614497,0.582731,0.298429,0.507657,0.506726,0.300628,0.506142,0.363814,0.298635
Poly Bridge,0.029182,0.298665,1.000000,0.010593,0.345824,0.868171,0.345688,0.672527,0.419135,0.522185,...,0.267745,0.474811,0.342849,0.671790,0.017570,0.015113,0.019256,0.867730,0.018070,0.671901
SolForge,0.504135,0.257014,0.010593,1.000000,0.297074,0.257834,0.012563,0.012059,0.012419,0.229967,...,0.451987,0.208469,0.009578,0.295800,0.505130,0.504799,0.297099,0.009293,0.012181,0.295948
Dishonored,0.590109,0.584054,0.345824,0.297074,1.000000,0.302262,0.022381,0.021475,0.022404,0.268838,...,0.523328,0.474801,0.344529,0.018434,0.586220,0.584714,0.348691,0.583815,0.421059,0.018830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Mass Effect,0.510655,0.506726,0.015113,0.504799,0.584714,0.261497,0.301289,0.017209,0.364645,0.453513,...,0.674840,0.614448,0.583017,0.299129,0.999757,1.000000,0.867813,0.259930,0.711973,0.299385
L.A. Noire,0.035667,0.300628,0.019256,0.297099,0.348691,0.302492,0.348504,0.021859,0.421839,0.522961,...,0.523304,0.474710,0.671976,0.345690,0.868763,0.867813,1.000000,0.300259,0.820439,0.345959
The Forest,0.025410,0.506142,0.867730,0.009293,0.583815,0.753349,0.300031,0.583621,0.363602,0.453075,...,0.453484,0.614516,0.582562,0.582947,0.261384,0.259930,0.300259,1.000000,0.363541,0.583064


### Item-Based

In [41]:
def item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbors=20, top_n=20):
    if user_id not in user_item_matrix.index:
        return None

    user_games = user_item_matrix.loc[user_id]
    played = user_games[user_games != 0]
    played_known = played[played.index.isin(item_sim_df.index)]

    unplayed = user_games[user_games == 0].index
    unplayed_known = [g for g in unplayed if g in item_sim_df.index]

    if len(played_known) == 0 or len(unplayed_known) == 0:
        return None

    scores = {}
    for game in unplayed_known:
        sim_scores = item_sim_df.loc[[game], played_known.index].iloc[0]
        top_neighbors = sim_scores.sort_values(ascending=False).head(n_neighbors)

        weights = top_neighbors.values
        ratings = played_known[top_neighbors.index].values

        denominator = weights.sum()
        if denominator > 0:
            scores[game] = np.dot(weights, ratings) / denominator

    result = pd.Series(scores).sort_values(ascending=False).head(top_n).reset_index()
    result.columns = ['game', 'ib_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

ib_recs = item_based_rec(sample_user, user_item_matrix, item_sim_df)

played, ib_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                           game  ib_score
 0                   I am Bread  3.395612
 1   Thinking with Time Machine  3.395396
 2                         Gish  3.376007
 3                      Rochard  3.367651
 4     Squishy the Suicidal Pig  3.367551
 5                     Bad Rats  3.367111
 6                     The Cave  3.360554
 7          Out There Somewhere  3.358990
 8       Woodle Tree Adventures  3.356704
 9             Thomas Was Alone  3.355856
 10                      Unepic  3.355550
 11              SteamWorld Dig  3.355028
 12           They Bleed Pixels  3.354876
 13                       Trine  3.353273
 14                     Trine 2  3.351712
 15         BattleBlock Theater  3.350606
 16                Rogue Legacy  3.349099
 17  

### Final Model (User-Based + Item-Based)

In [43]:
# alpha - вес User-Based оценки
# beta - вес Item-Based оценки
# boost_factor - множитель для игр из пересечения User-Based и Item-Based
# discount_factor - множитель для игр только из одного

def hybrid_rec(user_id, user_item_matrix, item_sim_df, n_neighbours=20, top_n=20,
               alpha=0.6, beta=0.4, boost_factor=1.3, discount_factor=0.8):
    ub = user_based_rec(user_id, user_item_matrix, n_neighbours, top_n)
    ib = item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbours, top_n)

    if ub is None or ib is None:
        return None

    def normalize(series):
        mn, mx = series.min(), series.max()

        return (series - mn) / (mx - mn + 1e-9)

    ub['ub_score'] = normalize(ub['ub_score'])
    ib['ib_score'] = normalize(ib['ib_score'])

    merged = pd.merge(ub, ib, on='game', how='outer')

    both = merged.dropna(subset=['ub_score', 'ib_score']).copy()
    both['final_score'] = (alpha * both['ub_score'] + beta * both['ib_score']) * boost_factor

    only_ub = merged[merged['ib_score'].isna()].copy()
    only_ub['final_score'] = only_ub['ub_score'] * discount_factor

    only_ib = merged[merged['ub_score'].isna()].copy()
    only_ib['final_score'] = only_ib['ib_score'] * discount_factor

    result = pd.concat([both, only_ub, only_ib], ignore_index=True)
    result = result[['game', 'final_score']].sort_values('final_score', ascending=False).head(top_n).reset_index(drop=True)

    result.index += 1

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

hybrid_recs = hybrid_rec(sample_user, user_item_matrix, item_sim_df)

played, hybrid_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                            game  final_score
 1                              Source Filmmaker     0.800000
 2                                    I am Bread     0.800000
 3                    Thinking with Time Machine     0.796939
 4                          Farming Simulator 15     0.553191
 5                                          Gish     0.521282
 6                          Realm of the Mad God     0.493617
 7                                Clicker Heroes     0.455319
 8                                       Rochard     0.402491
 9                      Squishy the Suicidal Pig     0.401065
 10                                     Bad Rats     0.394818
 11                           Tabletop Simulator     0.387234
 12              